# Lesson 13 Lab — Matmul Tiling and the Library Boundary

**Puzzle:** When block tiles, tl.dot, L2 reuse, and cuBLAS baselines change together, which observation tells you whether the kernel, layout, toolchain, or hardware boundary is responsible?

This notebook retains one complete RTX 5090 execution.


## Why this matters

This lab isolates block tiles, tl.dot, L2 reuse, and cuBLAS baselines and keeps its comparison path explicit.


## 0. Predict before running

Predict correctness, warm latency ordering, and the first boundary case. Write what would disprove each prediction.


## 1. Theory and mechanism

A Triton GEMM assigns an output tile to each program, advances through K tiles, accumulates in FP32, and stores once. Tile sizes shape tensor-core use, reuse, and occupancy. Vendor libraries remain the control because standard GEMM is their strongest territory.


## 2. Trace the mechanism

```mermaid
flowchart LR
  A["Frozen input + contract"] --> B["block tiles, tl.dot, L2 reuse, and cuBLAS baselines"]
  B --> C["Triton candidate"]
  B --> D["CUDA / library control"]
  C --> E["correctness + samples"]
  D --> E
  E --> F["bounded decision"]
```


## 3. Inspect the comparison boundary

Baseline: named PyTorch CUDA/library or standard-grid path. Candidate: reviewed Triton kernel or explicit model described below.

One square shape cannot establish parity with cuBLAS across skinny, batched, small, and awkward matrices.


## 4. Inspect the execution environment

The next cell asserts CUDA and records GPU, target, PyTorch, CUDA runtime, Triton, Python, and seed.


In [1]:
from pathlib import Path
import json, sys

ROOT = Path.cwd().parents[2]
sys.path.insert(0, str(ROOT / "scripts"))
from chapter05_runtime import environment, run_lesson

LESSON_NO = 13
LESSON_TITLE = 'Matmul Tiling and the Library Boundary'
ENV = environment(LESSON_NO)
print(json.dumps(ENV, indent=2, ensure_ascii=False))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.13.0+cu130",
  "cuda_runtime": "13.0",
  "triton": "3.7.1",
  "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
  "python": "3.12.3",
  "seed": 20260826
}


## 5. Freeze the experiment

**Experiment:** Compare a compact teaching Triton FP16 GEMM with torch.mm on one frozen 512-square shape.

Inputs, output contract, timer, and target stay fixed across compared paths.


## 6. Inspect and execute the reviewed code

The next cell calls the shared reviewed kernel source, retains full samples in `metrics`, checks maximum error, and prints the bounded analysis.


In [2]:
metrics, analysis_en, analysis_zh = run_lesson(LESSON_NO)
print(json.dumps(metrics, indent=2, ensure_ascii=False))
print(analysis_en)


{
  "primary": 0.022911999374628067,
  "secondary": 0.014560000039637089,
  "max_abs_error": 0.0,
  "passed": true,
  "details": {
    "triton_tflops": 11.715933280674575,
    "library_tflops": 18.436501048710902,
    "dtype": "torch.float16",
    "fused_relu": false,
    "triton_samples_ms": [
      0.037696000188589096,
      0.028063999488949776,
      0.02707199938595295,
      0.024768000468611717,
      0.023423999547958374,
      0.02643200010061264,
      0.02287999913096428,
      0.02412799932062626,
      0.02287999913096428,
      0.022784000262618065,
      0.022911999374628067,
      0.021568000316619873,
      0.021856000646948814,
      0.0225600004196167,
      0.022336000576615334
    ],
    "library_samples_ms": [
      0.01679999940097332,
      0.015072000212967396,
      0.015200000256299973,
      0.014336000196635723,
      0.014495999552309513,
      0.014015999622642994,
      0.015039999969303608,
      0.014592000283300877,
      0.014015999622642994,
      

## 7. Read the retained RTX 5090 result

**Environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.13.0+cu130; CUDA runtime 13.0; Triton 3.7.1; Python 3.12.3.

| Measured field | Checked-in value |
|---|---:|
| Triton median | 0.0229 ms |
| torch.mm median | 0.0146 ms |
| Maximum absolute error | 0.000e+00 |
| Acceptance gate | true |


## 8. Explain without overclaiming

The teaching Triton GEMM reached 11.72 TFLOP/s; torch.mm reached 18.44. The library remains the baseline, not a guaranteed loser.

A named Triton or PyTorch CUDA path executed on the recorded GPU. The result applies to the printed shape, dtype, implementation, and software stack; internal hardware causes require profiler evidence.


## 9. Write the canonical artifact

The next cell stores the environment, full metrics, bilingual analysis, evidence label, and bounded conclusion.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": LESSON_NO,
    "title": LESSON_TITLE,
    "environment": ENV,
    "evidence_label": 'native-backend',
    "metrics": metrics,
    "analysis_en": analysis_en,
    "analysis_zh": analysis_zh,
    "conclusion": 'Prefer the library for plain GEMM unless custom layout or fused epilogue value survives a representative shape sweep.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 13,
  "title": "Matmul Tiling and the Library Boundary",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.13.0+cu130",
    "cuda_runtime": "13.0",
    "triton": "3.7.1",
    "triton_target": "GPUTarget(backend='cuda', arch=120, warp_size=32)",
    "python": "3.12.3",
    "seed": 20260826
  },
  "evidence_label": "native-backend",
  "metrics": {
    "primary": 0.022911999374628067,
    "secondary": 0.014560000039637089,
    "max_abs_error": 0.0,
    "passed": true,
    "details": {
      "triton_tflops": 11.715933280674575,
      "library_tflops": 18.436501048710902,
      "dtype": "torch.float16",
      "fused_relu": false,
      "triton_samples_ms": [
        0.037696000188589096,
        0.028063999488949776,
        0.02707199938595295,
        0.024768000468611717,
        0.023423999547958374,
        0.02643200010061264,
        0.02287999913096428,
        0.02412799932062626,
        0.02287999913096428,
  

## 10. Make the bounded decision

> Prefer the library for plain GEMM unless custom layout or fused epilogue value survives a representative shape sweep.

**Failure analysis:** One square shape cannot establish parity with cuBLAS across skinny, batched, small, and awkward matrices.


## 11. Extend and review

Add an awkward shape and non-contiguous layout. Stop on correctness failure. See `README.md` for references and the full review checklist.
